# F

Dit is allemaal heel mooi en absoluut zelf geschreven, zeken geen AI aan te pas gekomen. \
Als je me een leugenaar noemt klaag ik je aan voor racisme.

In [3]:
import pandas as pd
import numpy as np

In [7]:
transaction_train = pd.read_csv('../../MLOps Data/train_transaction.csv')
transaction_test = pd.read_csv('../../MLOps Data/test_transaction.csv')

identity_train = pd.read_csv('../../MLOps Data/train_identity.csv')
identity_test = pd.read_csv('../../MLOps Data/test_identity.csv')

# Edge Model

In [8]:
from sklearn.ensemble import IsolationForest

# Edge Model: Simple anomaly detection using IsolationForest

# Select a few relevant numeric features for the edge model
edge_features = ['TransactionAmt', 'card1', 'card2', 'card3', 'card5', 'addr1', 'addr2']
X_edge = transaction_train[edge_features].fillna(-999)

# Train IsolationForest as a lightweight anomaly detector
edge_model = IsolationForest(n_estimators=50, contamination=0.01, random_state=42)
edge_model.fit(X_edge)

# Predict anomalies on new transactions (simulate edge device)
X_test_edge = transaction_test[edge_features].fillna(-999)
edge_preds = edge_model.predict(X_test_edge)  # -1 = anomaly, 1 = normal

# Mark suspicious transactions for cloud validation
transaction_test['edge_flag'] = edge_preds
suspicious = transaction_test[transaction_test['edge_flag'] == -1]

# Simulate API call: send suspicious transactions to cloud model
# (Here, just print how many would be sent)
print(f"Number of transactions flagged as suspicious by edge model: {len(suspicious)}")

Number of transactions flagged as suspicious by edge model: 7426


# Cloud Model

In [9]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.utils import to_categorical

# Select features and target for the cloud model
cloud_features = ['TransactionAmt', 'card1', 'card2', 'card3', 'card5', 'addr1', 'addr2']
X = transaction_train[cloud_features].fillna(-999)
y = transaction_train['isFraud']

# Split data for training and validation
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

# Build a simple deep learning model
model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dropout(0.2),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train the model
model.fit(X_train_scaled, y_train, epochs=5, batch_size=256, validation_data=(X_val_scaled, y_val))

# Example: Predict on suspicious transactions (from edge)
X_suspicious = suspicious[cloud_features].fillna(-999)
X_suspicious_scaled = scaler.transform(X_suspicious)
cloud_preds = model.predict(X_suspicious_scaled)
suspicious['cloud_fraud_prob'] = cloud_preds

# Mark as fraud if probability > 0.5
suspicious['cloud_flag'] = (suspicious['cloud_fraud_prob'] > 0.5).astype(int)

print(suspicious[['TransactionID', 'cloud_fraud_prob', 'cloud_flag']].head())




Epoch 1/5


1846/1846 [==============================] - 4s 2ms/step - loss: 0.1562 - accuracy: 0.9636 - val_loss: 0.1392 - val_accuracy: 0.9652
Epoch 2/5
1846/1846 [==============================] - 3s 2ms/step - loss: 0.1431 - accuracy: 0.9651 - val_loss: 0.1389 - val_accuracy: 0.9653
Epoch 3/5
1846/1846 [==============================] - 3s 2ms/step - loss: 0.1417 - accuracy: 0.9652 - val_loss: 0.1384 - val_accuracy: 0.9652
Epoch 4/5
1846/1846 [==============================] - 3s 2ms/step - loss: 0.1406 - accuracy: 0.9652 - val_loss: 0.1380 - val_accuracy: 0.9653
Epoch 5/5
233/233 [==============================] - 0s 647us/step
     TransactionID  cloud_fraud_prob  cloud_flag
309        3663858          0.030923           0
360        3663909          0.208107           0
471        3664020          0.113161           0
687        3664236          0.065208           0
879        3664428          0.029639           0


C:\Users\tijnw\AppData\Local\Temp\ipykernel_31496\2470372096.py:38: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  suspicious['cloud_fraud_prob'] = cloud_preds
C:\Users\tijnw\AppData\Local\Temp\ipykernel_31496\2470372096.py:41: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  suspicious['cloud_flag'] = (suspicious['cloud_fraud_prob'] > 0.5).astype(int)
